In [ ]:
import os
# Set working directory and GPU before importing JAX
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import vmap

from jaxpi.models import (
    create_lr_schedule,
    create_optimizer,
    create_arch,
    create_train_state,
)
from jaxpi.checkpointing import (
    create_checkpoint_manager,
    save_checkpoint,
    restore_checkpoint,
)

import models
from utils import get_dataset

jax.config.update("jax_default_matmul_precision", "highest")

In [ ]:
# Load config
from configs import baseline, fixed_pseudo_time, pseudo_time
config = baseline.get_config()

In [ ]:
# Get dataset
u_ref, t_star, x_star = get_dataset(time_range=config.time_range)
u0 = u_ref[0, :]

In [ ]:
# Initialize model
lr = create_lr_schedule(config.optim)
tx = create_optimizer(config.optim, lr)
arch = create_arch(config.arch)
state = create_train_state(config, tx, arch)
model = models.KS(config, lr, tx, arch, state, u0, t_star, x_star)

In [ ]:
# Create checkpoint manager
window_idx = 1
ckpt_path = os.path.join(os.getcwd(), config.wandb.name, "ckpt", 'time_window_{}'.format(window_idx))
ckpt_mngr = create_checkpoint_manager(config.saving, ckpt_path)

# restore checkpoint
state = restore_checkpoint(ckpt_mngr, state)
print("Restore checkpoint from step: ", int(state.step))

In [ ]:
# Evaluate on test set
u_pred = vmap(vmap(model.neural_net, (None, None, 0)), (None, 0, None))(state.params, model.t_star, model.x_star)
error = jnp.linalg.norm(u_pred - u_ref) / jnp.linalg.norm(u_ref)
print("Relative L2 error: {:.2e}".format(error))

In [ ]:
# Plot results
plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
plt.title("Reference solution")
plt.pcolor(model.t_star, model.x_star, u_ref.T, shading="auto", cmap="jet")
plt.colorbar()
plt.subplot(1, 3, 2)
plt.title("Predicted solution")
plt.pcolor(model.t_star, model.x_star, u_pred.T, shading="auto", cmap="jet")
plt.colorbar()
plt.subplot(1, 3, 3)
plt.title("Absolute error")
plt.pcolor(model.t_star, model.x_star, jnp.abs(u_pred - u_ref).T, shading="auto", cmap="jet")
plt.colorbar()
plt.tight_layout()
plt.show()